<!-- torchleet:colab -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fengyee/TorchLeet/blob/main/v4/dl-fundamentals/mlp-backprop/mlp-backprop.ipynb)

Check your work: `pip install -e python/` once from the repo root (on Colab: `!pip install -q "git+https://github.com/Fengyee/TorchLeet#subdirectory=python"`), then `from torchleet import check; check("mlp-backprop", ...)`


# MLP Backprop by Hand (No Autograd)

**Difficulty**: 🟡 Medium

### Problem Statement

Every deep-learning framework is, underneath, the chain rule applied one layer at a time. Writing the backward pass of a small network by hand (the CS231n "two-layer net" exercise; Karpathy's *"Yes you should understand backprop"*) is a classic interview question because it exposes whether you really know which shapes flow backwards, what the forward pass must cache, and why softmax + cross-entropy has such a simple gradient. It is also the mental model you need to debug exploding/vanishing gradients, write a custom `torch.autograd.Function`, or reason about activation memory and gradient checkpointing.

Build an MLP classifier from three hand-differentiated pieces — `Linear`, `ReLU` and a fused softmax-cross-entropy loss — and train it with plain SGD on a two-spirals toy dataset. **No autograd anywhere.**

---

### Requirements

1. **`Linear(in_features, out_features)`** — a plain Python class (not an `nn.Module`).
   - Parameters are plain tensors `self.W` of shape `(out_features, in_features)` (the same layout as `nn.Linear.weight`) and `self.b` of shape `(out_features,)`.
   - **He/Kaiming-uniform init:** $W \sim \mathcal{U}(-a, a)$ with $a = \sqrt{6/d_\text{in}}$ (where $d_\text{in}$ = `in_features` = fan-in), so that $\mathrm{Var}(W) = 2/d_\text{in}$; $b = 0$. (The grader checks the zero mean and the variance $2/d_\text{in}$, so He-normal also passes.)
   - `forward(x)`: `x` is `(N, in_features)`; return $y = xW^\top + b$, shape `(N, out_features)`, and cache whatever `backward` needs.
   - `backward(grad_out)`: `grad_out` is $\partial L/\partial y$, shape `(N, out_features)`. Store `self.grad_W` `(out, in)` and `self.grad_b` `(out,)` — **overwrite, don't accumulate** — and return $\partial L/\partial x$, shape `(N, in_features)`.
   - Read `self.W` / `self.b` at call time: the grader assigns its own (float64) tensors to them.
2. **`ReLU()`** — `forward(x)` returns $\max(x, 0)$; `backward(grad_out)` returns $\partial L/\partial x$. At exactly $x = 0$ the gradient is **0** (PyTorch's convention: the local gradient is $\mathbb{1}[x > 0]$).
3. **`softmax_cross_entropy(logits, targets) -> (loss, grad_logits)`**
   - `logits`: `(N, C)`; `targets`: `(N,)` int64 class indices.
   - `loss`: scalar tensor, the batch **mean** of $-\log \mathrm{softmax}(z_n)_{y_n}$ — identical to `F.cross_entropy(logits, targets)`.
   - `grad_logits`: $\partial\,\text{loss} / \partial z = \big(\mathrm{softmax}(z) - \mathrm{onehot}(y)\big)/N$, shape `(N, C)`.
   - Numerically stable: finite and correct for logits of magnitude $10^3$ (exact float64 agreement with `F.cross_entropy`).
4. **`MLP(sizes)`** — e.g. `MLP([2, 32, 32, 2])` is `Linear(2,32) → ReLU → Linear(32,32) → ReLU → Linear(32,2)`; **no ReLU on the logits**.
   - Keep the layers, in forward order, in a list `self.layers`.
   - `forward(x)` → logits `(N, sizes[-1])`.
   - `backward(grad_logits)` → runs every layer's `backward` in reverse order and returns $\partial L/\partial x$.
   - `step(lr)` → plain SGD on every `Linear`: `W -= lr * grad_W`, `b -= lr * grad_b`.
5. Everything must work in float32 and float64 under `torch.no_grad()` on tensors with `requires_grad=False` — that is how the grader runs it.

---

### Constraints

- ✅ Plain tensor ops: `@`, `.T`, `.sum`, `.max`, `.exp`, `.log`, `.clamp`, advanced indexing, `torch.arange`, `torch.logsumexp`.
- ❌ No autograd inside your code: no `.backward()`, `torch.autograd.*`, `requires_grad_()`.
- ❌ No `nn.Linear`, `F.cross_entropy`, `F.softmax` / `F.log_softmax`, `torch.optim` in the implementation (the tests use them as the oracle).

---

<details>
  <summary>💡 Hint</summary>

  - Draw the shapes. A gradient always has exactly the shape of the tensor it is the gradient of: `grad_W` is `(out, in)`. Which product of `grad_out (N, out)` and `x (N, in)` has that shape?
  - The bias is added to every row of the batch. What must happen to the $N$ per-row contributions in the backward pass?
  - For the loss, write $\ell = \log\sum_j e^{z_j} - z_y$ and differentiate w.r.t. $z_k$ directly — you never need the full softmax Jacobian.
  - $\log\sum_j e^{z_j} = m + \log\sum_j e^{z_j - m}$ for any $m$. Which $m$ makes every exponent $\le 0$?
  - The $1/N$ of the mean belongs to the loss gradient only — the layers just apply the chain rule to whatever arrives.

</details>

---

In [ ]:
import math
import torch
import torch.nn.functional as F   # used by the tests as an oracle, never inside your implementation

In [ ]:
def make_spirals(n_per_class=100, noise=0.15, turns=1.25, seed=0):
    """Two interleaved spirals in [-1, 1]^2 — a classic not-linearly-separable toy set.

    Returns:
        X: (2 * n_per_class, 2) float32 points
        y: (2 * n_per_class,) int64 labels in {0, 1}
    """
    g = torch.Generator().manual_seed(seed)
    X, y = [], []
    for c in range(2):
        r = torch.linspace(0.1, 1.0, n_per_class)
        t = (torch.linspace(0, turns * 2 * math.pi, n_per_class) + c * math.pi
             + noise * torch.randn(n_per_class, generator=g))
        X.append(torch.stack([r * torch.cos(t), r * torch.sin(t)], dim=1))
        y.append(torch.full((n_per_class,), c, dtype=torch.long))
    return torch.cat(X), torch.cat(y)


X, y = make_spirals()
print(f"X {tuple(X.shape)}, y {tuple(y.shape)}, class counts {y.bincount().tolist()}")

In [ ]:
class Linear:
    """Fully connected layer y = x @ W.T + b with a hand-written backward pass."""

    def __init__(self, in_features: int, out_features: int):
        """
        Sets:
            self.W: (out_features, in_features), He-uniform U(-sqrt(6/in), sqrt(6/in))
            self.b: (out_features,), zeros
            self.grad_W, self.grad_b: None until backward() runs
        """
        # TODO
        raise NotImplementedError

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (N, in_features)
        Returns:
            (N, out_features). Cache what backward() needs.
        """
        # TODO
        raise NotImplementedError

    def backward(self, grad_out: torch.Tensor) -> torch.Tensor:
        """
        Args:
            grad_out: dL/dy, (N, out_features)
        Returns:
            dL/dx, (N, in_features). Also sets self.grad_W (out, in) and self.grad_b (out,),
            overwriting any previous values.
        """
        # TODO
        raise NotImplementedError

    def __call__(self, x):
        return self.forward(x)


class ReLU:
    """max(x, 0) with a hand-written backward pass (gradient 0 at x == 0, like PyTorch)."""

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: any shape. Returns max(x, 0), same shape. Cache what backward() needs."""
        # TODO
        raise NotImplementedError

    def backward(self, grad_out: torch.Tensor) -> torch.Tensor:
        """grad_out: dL/dy, same shape as x. Returns dL/dx."""
        # TODO
        raise NotImplementedError

    def __call__(self, x):
        return self.forward(x)

In [ ]:
def softmax_cross_entropy(logits: torch.Tensor, targets: torch.Tensor):
    """Mean softmax cross-entropy and its gradient w.r.t. the logits.

    Args:
        logits:  (N, C) unnormalized scores
        targets: (N,) int64 class indices in [0, C)
    Returns:
        loss:        scalar tensor, mean over the batch of -log softmax(logits)[n, targets[n]]
        grad_logits: (N, C), d loss / d logits = (softmax(logits) - onehot(targets)) / N
    Must stay finite and exact for logits of magnitude 1e3.
    """
    # TODO
    raise NotImplementedError

In [ ]:
class MLP:
    """Linear -> ReLU -> ... -> Linear (no ReLU on the logits), trained with plain SGD."""

    def __init__(self, sizes):
        """
        Args:
            sizes: layer widths, e.g. [2, 32, 32, 2] = 3 Linear layers.
        Sets:
            self.layers: list of Linear / ReLU objects in forward order.
        """
        # TODO
        raise NotImplementedError

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (N, sizes[0]) -> logits (N, sizes[-1])."""
        # TODO
        raise NotImplementedError

    def backward(self, grad: torch.Tensor) -> torch.Tensor:
        """grad: dL/dlogits (N, sizes[-1]). Fills every Linear's grad_W / grad_b; returns dL/dx (N, sizes[0])."""
        # TODO
        raise NotImplementedError

    def step(self, lr: float) -> None:
        """Plain SGD: W -= lr * grad_W and b -= lr * grad_b for every Linear layer."""
        # TODO
        raise NotImplementedError

    def __call__(self, x):
        return self.forward(x)

In [ ]:
# Test 1 — Linear: output and all three gradients vs torch.autograd (float64)
torch.manual_seed(0)
lin = Linear(5, 4)
assert tuple(lin.W.shape) == (4, 5) and tuple(lin.b.shape) == (4,), "W must be (out, in), b (out,)"
lin.W = torch.randn(4, 5, dtype=torch.float64)
lin.b = torch.randn(4, dtype=torch.float64)
x = torch.randn(7, 5, dtype=torch.float64)
g = torch.randn(7, 4, dtype=torch.float64)            # an arbitrary upstream gradient dL/dy

with torch.no_grad():                                  # your code must not need autograd
    out = lin.forward(x)
    gx = lin.backward(g)

W_ref, b_ref, x_ref = (t.clone().requires_grad_() for t in (lin.W, lin.b, x))
F.linear(x_ref, W_ref, b_ref).backward(g)

assert torch.allclose(out, F.linear(x, lin.W, lin.b)), "forward is wrong"
assert torch.allclose(lin.grad_W, W_ref.grad), "grad_W is wrong"
assert torch.allclose(lin.grad_b, b_ref.grad), "grad_b is wrong"
assert torch.allclose(gx, x_ref.grad), "dL/dx is wrong"
print("Linear: forward and gradients match autograd")

In [ ]:
# Test 2 — ReLU, including the gradient at exactly 0
relu = ReLU()
x = torch.tensor([-2.0, -0.5, 0.0, 0.5, 3.0], dtype=torch.float64)
with torch.no_grad():
    out = relu.forward(x)
    gx = relu.backward(torch.ones_like(x))
assert torch.allclose(out, torch.tensor([0.0, 0.0, 0.0, 0.5, 3.0], dtype=torch.float64))
assert torch.allclose(gx, torch.tensor([0.0, 0.0, 0.0, 1.0, 1.0], dtype=torch.float64)), \
    f"got {gx.tolist()} — the gradient at exactly 0 must be 0 (PyTorch's convention)"
print("ReLU OK")

In [ ]:
# Test 3 — softmax cross-entropy vs F.cross_entropy, on normal and extreme logits
torch.manual_seed(0)
cases = {
    "random": (3 * torch.randn(6, 5, dtype=torch.float64), torch.randint(0, 5, (6,))),
    "extreme": (torch.tensor([[1000.0, -1000.0, 0.0],
                              [-1000.0, 1000.0, 1000.0],
                              [999.0, 1000.0, -1000.0]], dtype=torch.float64),
                torch.tensor([1, 0, 0])),
}
for name, (z, t) in cases.items():
    with torch.no_grad():
        loss, grad = softmax_cross_entropy(z, t)
    z_ref = z.clone().requires_grad_()
    ref = F.cross_entropy(z_ref, t)
    ref.backward()
    assert torch.isfinite(torch.as_tensor(loss)), f"{name}: loss is {loss}"
    assert torch.allclose(torch.as_tensor(loss, dtype=torch.float64), ref.detach()), \
        f"{name}: loss {float(loss):.6f} vs F.cross_entropy {ref.item():.6f}"
    assert torch.allclose(grad, z_ref.grad), f"{name}: grad_logits is wrong"
    print(f"{name:8s} logits: loss {float(loss):10.4f} — matches F.cross_entropy, gradient OK")

In [ ]:
# Test 4 — whole MLP: every gradient vs autograd on an equivalent graph (float64)
torch.manual_seed(0)
mlp = MLP([6, 9, 7, 4])
linears = [layer for layer in mlp.layers if isinstance(layer, Linear)]
assert len(linears) == 3
for layer in linears:                                  # float64 for a tight comparison
    layer.W, layer.b = layer.W.double(), (0.1 * torch.randn_like(layer.b)).double()
x = torch.randn(10, 6, dtype=torch.float64)
t = torch.randint(0, 4, (10,))

with torch.no_grad():
    logits = mlp.forward(x)
    loss, grad_logits = softmax_cross_entropy(logits, t)
    gx = mlp.backward(grad_logits)

params = [p.clone().requires_grad_() for layer in linears for p in (layer.W, layer.b)]
x_ref = x.clone().requires_grad_()
h = x_ref
for k in range(len(linears)):
    h = F.linear(h, params[2 * k], params[2 * k + 1])
    if k < len(linears) - 1:
        h = F.relu(h)
loss_ref = F.cross_entropy(h, t)
loss_ref.backward()

assert torch.allclose(logits, h.detach()), "MLP forward is wrong (is there a ReLU on the logits?)"
for k, layer in enumerate(linears):
    assert torch.allclose(layer.grad_W, params[2 * k].grad), f"grad_W of layer {k} is wrong"
    assert torch.allclose(layer.grad_b, params[2 * k + 1].grad), f"grad_b of layer {k} is wrong"
assert torch.allclose(gx, x_ref.grad), "dL/dx returned by MLP.backward is wrong"
print("MLP: all gradients match autograd")

In [ ]:
# Test 5 — train on the two spirals with full-batch SGD
torch.manual_seed(0)
model = MLP([2, 32, 32, 2])
losses = []
with torch.no_grad():
    for step in range(300):
        loss, grad = softmax_cross_entropy(model.forward(X), y)
        model.backward(grad)
        model.step(lr=0.5)
        losses.append(float(loss))
    acc = (model.forward(X).argmax(dim=1) == y).float().mean().item()
print(f"loss {losses[0]:.3f} -> {losses[-1]:.4f}, train accuracy {acc:.1%}")
assert losses[-1] < 0.3 * losses[0] and acc > 0.95, "the MLP is not learning the spirals"

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("mlp-backprop", Linear=Linear, ReLU=ReLU, softmax_cross_entropy=softmax_cross_entropy, MLP=MLP)